# Capítulo 3 · NumPy desde cero

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riosinda/ml-course/blob/main/notebooks/lectures/chapter_03/01_numpy.ipynb)

**Fundamentos para Machine Learning**

---

### ¿Por qué empezar por NumPy?

En Machine Learning **todo se reduce a números organizados en arreglos**:

| Dato del mundo real | Cómo lo ve un modelo de ML |
|---|---|
| Un dataset tabular de 1000 clientes con 5 variables | matriz de forma `(1000, 5)` |
| Una imagen en escala de grises de 28×28 píxeles | matriz de forma `(28, 28)` |
| Un lote de 32 fotos a color de 64×64 | tensor de forma `(32, 64, 64, 3)` |
| Los pesos de una capa de una red neuronal | matriz de forma `(n_entradas, n_salidas)` |

**NumPy** (*Numerical Python*) es la librería que permite trabajar con estos arreglos de forma rápida y cómoda. Es la base sobre la que se construyen **pandas**, **scikit-learn** y **matplotlib**, y su forma de trabajar inspiró directamente a **PyTorch** y **TensorFlow**. Si dominas NumPy, cualquier otra herramienta de ML te resultará familiar.

### 🎯 Objetivos

Al terminar este notebook serás capaz de:
1. Crear y manipular arrays de cualquier dimensión.
2. Indexar, filtrar y cambiar la forma de los datos.
3. Escribir código **vectorizado** (sin bucles).
4. Resumir datos con agregaciones y el parámetro `axis`.

### 📚 Índice
1. ¿Qué es un array y por qué no usar listas?
2. Crear arrays
3. Atributos y tipos de datos (`dtype`)
4. Indexación, *slicing* y máscaras booleanas
5. Cambiar la forma de los arrays
6. Operaciones vectorizadas
7. Agregaciones y el parámetro `axis`

In [2]:
import numpy as np

print("Versión de NumPy:", np.__version__)

Versión de NumPy: 2.5.3


> 💡 `np` es el alias universal de NumPy. En cualquier código de ML que leas verás `import numpy as np`.

---
## 1. ¿Qué es un array y por qué no usar listas?

Un **array** (`ndarray`) es una colección de elementos **del mismo tipo** guardados de forma contigua en memoria. Parece una lista de Python, pero se comporta distinto:

In [3]:
lista = [1, 2, 3, 4]
arr = np.array([1, 2, 3, 4])

print("lista * 2 →", lista * 2)   # repite la lista
print("arr * 2   →", arr * 2)     # multiplica cada elemento

print("lista + lista →", lista + lista)  # concatena
print("arr + arr     →", arr + arr)      # suma elemento a elemento

lista * 2 → [1, 2, 3, 4, 1, 2, 3, 4]
arr * 2   → [2 4 6 8]
lista + lista → [1, 2, 3, 4, 1, 2, 3, 4]
arr + arr     → [2 4 6 8]


Con NumPy las operaciones matemáticas se aplican **elemento a elemento**, que es justo lo que necesitamos al trabajar con datos.

Además, NumPy es **muchísimo más rápido** porque las operaciones se ejecutan en código C optimizado, no en un bucle de Python. Comprobémoslo:

In [4]:
import time

n = 1_000_000
lista = list(range(n))
arr = np.arange(n)

t0 = time.perf_counter()
cuadrados_lista = [x ** 2 for x in lista]
t_lista = time.perf_counter() - t0

t0 = time.perf_counter()
cuadrados_arr = arr ** 2
t_arr = time.perf_counter() - t0

print(f"Lista de Python: {t_lista * 1000:7.1f} ms")
print(f"Array de NumPy:  {t_arr * 1000:7.1f} ms")
print(f"→ NumPy es ~{t_lista / t_arr:.0f} veces más rápido")

Lista de Python:   103.5 ms
Array de NumPy:      5.7 ms
→ NumPy es ~18 veces más rápido


Los **f-strings** (o *Formatted String Literals*) son una característica introducida en Python 3.6 que permite incrustar expresiones, variables y cálculos directamente dentro de cadenas de texto de una manera concisa y fácil de leer.

Para crear un f-string, simplemente se coloca la letra `f` o `F` justo antes de las comillas de apertura (pueden ser simples `' '` o dobles `" "`). Todo lo que se encuentre dentro de llaves `{}` será evaluado por Python y convertido a texto.

### Características principales y ejemplos

**1. Insertar variables directamente:**
En lugar de concatenar con `+` o usar comas, simplemente colocas el nombre de la variable entre llaves.

```python
nombre = "Ana"
edad = 28
mensaje = f"Hola, me llamo {nombre} y tengo {edad} años."
# Resultado: "Hola, me llamo Ana y tengo 28 años."

```

**2. Evaluar operaciones y expresiones:**
Puedes realizar operaciones matemáticas o llamar funciones directamente dentro de las llaves.

```python
precio = 50
cantidad = 3
total = f"El costo total es: ${precio * cantidad}"
# Resultado: "El costo total es: $150"

```

**3. Aplicar formato numérico:**
Como vimos en tu código anterior (`{t_lista * 1000:7.1f}`), puedes usar los dos puntos `:` dentro de las llaves para especificar cómo quieres que se muestre un valor.

```python
pi = 3.14159265
# Limitar a 2 decimales (.2f)
print(f"El valor de pi es aproximadamente {pi:.2f}")
# Resultado: "El valor de pi es aproximadamente 3.14"

dinero = 1500
# Agregar separador de miles (,)
print(f"Tienes ${dinero:,} en tu cuenta")
# Resultado: "Tienes $1,500 en tu cuenta"

```

**4. Depuración rápida (Python 3.8+):**
Si agregas un signo de igual `=` después de la variable dentro de las llaves, el f-string imprimirá tanto el nombre de la variable como su valor. Esto es muy útil para encontrar errores.

```python
usuario = "Carlos"
print(f"{usuario=}")
# Resultado: "usuario='Carlos'"

```

Cuando entrenes un modelo con millones de datos, esta diferencia es la que separa minutos de horas. A escribir código sin bucles se le llama **vectorizar**, y es la habilidad más importante de este notebook.

---
## 2. Crear arrays

### 2.1 A partir de listas

In [ ]:
# 1 dimensión (vector)
v = np.array([3, 1, 4, 1, 5])
print(v)

# 2 dimensiones (matriz): una lista de listas
M = np.array([[1, 2, 3],
              [4, 5, 6]])
print(M)

# 3 dimensiones (tensor): por ejemplo, 2 imágenes de 2x3 píxeles
T = np.array([[[1, 2, 3], [4, 5, 6]],
              [[7, 8, 9], [10, 11, 12]]])
print(T)

(5,)
[[1 2 3]
 [4 5 6]]
[[[ 1  2  3]
  [ 4  5  6]]

 [[ 7  8  9]
  [10 11 12]]]


### 2.2 Arrays "prefabricados"

Muy habituales para inicializar pesos, crear máscaras o reservar memoria:

In [6]:
print("zeros:\n", np.zeros((2, 3)))        # todo ceros
print("ones:\n", np.ones((2, 3)))          # todo unos
print("full:\n", np.full((2, 3), 7))       # todo un valor
print("eye (identidad):\n", np.eye(3))     # matriz identidad

zeros:
 [[0. 0. 0.]
 [0. 0. 0.]]
ones:
 [[1. 1. 1.]
 [1. 1. 1.]]
full:
 [[7 7 7]
 [7 7 7]]
eye (identidad):
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


In [7]:
# Secuencias numéricas
print(np.arange(0, 10, 2))       # como range(): inicio, fin (excluido), paso
print(np.linspace(0, 1, 5))      # 5 valores equiespaciados entre 0 y 1 (ambos incluidos)

[0 2 4 6 8]
[0.   0.25 0.5  0.75 1.  ]


> 💡 **`arange` vs `linspace`**: usa `arange` cuando conoces el **paso**, y `linspace` cuando conoces **cuántos puntos** quieres. `linspace` es la favorita para graficar funciones (lo veremos al hacer gráficas).

In [8]:
# Crear arrays con la misma forma que otro
X = np.array([[1.5, 2.0], [3.1, 4.2]])
print(np.zeros_like(X))
print(np.ones_like(X))

[[0. 0.]
 [0. 0.]]
[[1. 1.]
 [1. 1.]]


---
## 3. Atributos y tipos de datos

Todo array tiene cuatro atributos que vas a consultar **constantemente** mientras depuras código de ML:

| Atributo | Qué indica |
|---|---|
| `.shape` | forma: tamaño de cada dimensión |
| `.ndim` | número de dimensiones |
| `.size` | número total de elementos |
| `.dtype` | tipo de dato de los elementos |

In [9]:
X = np.zeros((100, 4))   # imagina 100 muestras con 4 características cada una

print("shape:", X.shape)
print("ndim: ", X.ndim)
print("size: ", X.size)
print("dtype:", X.dtype)

shape: (100, 4)
ndim:  2
size:  400
dtype: float64


### 3.1 Tipos de datos (`dtype`)

In [10]:
a = np.array([1, 2, 3])
b = np.array([1.0, 2.0, 3.0])
c = np.array([1, 2.5, 3])         # mezcla → NumPy convierte todo a float
d = np.array([True, False, True])

for nombre, x in [("a", a), ("b", b), ("c", c), ("d", d)]:
    print(nombre, x, x.dtype)

a [1 2 3] int64
b [1. 2. 3.] float64
c [1.  2.5 3. ] float64
d [ True False  True] bool


In [11]:
# Convertir de tipo con astype
x = np.array([1.7, 2.2, 3.9])
print(x.astype(int))          # trunca los decimales
print(x.astype(np.float32))   # menos precisión, la mitad de memoria

[1 2 3]
[1.7 2.2 3.9]


> 🧠 **En ML**: las redes neuronales suelen trabajar en `float32` (o incluso `float16`) porque ocupa la mitad de memoria que `float64` y las GPUs son más rápidas con él. Comprobémoslo:

In [12]:
grande64 = np.zeros((1000, 1000), dtype=np.float64)
grande32 = np.zeros((1000, 1000), dtype=np.float32)
print(f"float64: {grande64.nbytes / 1e6:.0f} MB")
print(f"float32: {grande32.nbytes / 1e6:.0f} MB")

float64: 8 MB
float32: 4 MB


1e6 (Notación científica): En Python, esto significa 1 x 10⁶, lo que equivale a 1,000,000 (un millón).

---
## 4. Indexación, *slicing* y máscaras booleanas

### 4.1 Arrays de 1 dimensión
Funciona igual que con listas de Python: los índices empiezan en `0` y los negativos cuentan desde el final.

In [13]:
a = np.array([10, 20, 30, 40, 50, 60])

print(a[0])      # primer elemento
print(a[-1])     # último elemento
print(a[1:4])    # del índice 1 al 3 (el 4 no se incluye)
print(a[:3])     # los 3 primeros
print(a[::2])    # uno sí, uno no
print(a[::-1])   # invertido

10
60
[20 30 40]
[10 20 30]
[10 30 50]
[60 50 40 30 20 10]


### 4.2 Arrays de 2 dimensiones
La sintaxis es `M[filas, columnas]`. En ML, por convención:
- **filas = muestras** (cada ejemplo, cliente, imagen…)
- **columnas = características** o *features* (edad, precio, píxel…)

In [14]:
M = np.array([[ 1,  2,  3,  4],
              [ 5,  6,  7,  8],
              [ 9, 10, 11, 12]])

print("Elemento fila 1, columna 2:", M[1, 2])
print("Fila 0 completa:          ", M[0])        # equivale a M[0, :]
print("Columna 1 completa:       ", M[:, 1])
print("Submatriz:\n", M[0:2, 1:3])

Elemento fila 1, columna 2: 7
Fila 0 completa:           [1 2 3 4]
Columna 1 completa:        [ 2  6 10]
Submatriz:
 [[2 3]
 [6 7]]


### 4.3 Máscaras booleanas: filtrar datos ⭐

Esta es una de las herramientas más usadas en análisis de datos. Una comparación sobre un array produce un **array de booleanos**, que podemos usar para filtrar:

In [15]:
edades = np.array([15, 22, 37, 8, 45, 19, 63])

mascara = edades >= 18
print("Máscara:", mascara)
print("Mayores de edad:", edades[mascara])

# Directamente en una línea
print("Menores:", edades[edades < 18])

Máscara: [False  True  True False  True  True  True]
Mayores de edad: [22 37 45 19 63]
Menores: [15  8]


En NumPy, la expresión `edades[mascara]` es la acción que aplica el filtro. Funciona indicándole a Python que extraiga del arreglo original (`edades`) únicamente los elementos donde la variable `mascara` contenga el valor `True`.

* **La evaluación previa:** La línea `mascara = edades >= 18` evalúa cada número del arreglo para ver si es mayor o igual a 18. Esto genera internamente un arreglo de respuestas booleanas (`True` o `False`): `[False, True, True, False, True, True, True]`.


* **La filtración (`edades[mascara]`):** Al colocar la máscara dentro de los corchetes, NumPy superpone el arreglo de booleanos sobre los datos originales de `edades`.


* **El resultado:** NumPy solo devuelve los valores que coinciden con la posición de un `True`. Como el 15 y el 8 (en la primera y cuarta posición) coinciden con un `False`, se omiten. El resultado de `edades[mascara]` es un nuevo arreglo: `[22, 37, 45, 19, 63]`.

In [16]:
# np.where: "si se cumple, esto; si no, aquello"
categoria = np.where(edades >= 18, "adulto", "menor")
print(categoria)

['menor' 'adulto' 'adulto' 'menor' 'adulto' 'adulto' 'adulto']


In [17]:
X = np.array([[1.0, 2.0],
              [1.5, 1.8],
              [5.0, 8.0],
              [6.0, 9.0]])
y = np.array([0, 0, 1, 1])

print("Muestras de la clase 1:\n", X[y == 1])

Muestras de la clase 1:
 [[5. 8.]
 [6. 9.]]


---
## 5. Cambiar la forma de los arrays

### 5.1 `reshape`
Reorganiza los mismos datos en otra forma. El número total de elementos debe coincidir.

In [18]:
a = np.arange(12)
print(a.shape)

print(a.reshape(3, 4))
print(a.reshape(2, 6))
print(a.reshape(4, -1))   # -1 = "calcula tú esta dimensión"

(12,)
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
[[ 0  1  2  3  4  5]
 [ 6  7  8  9 10 11]]
[[ 0  1  2]
 [ 3  4  5]
 [ 6  7  8]
 [ 9 10 11]]


> 🧠 **En ML**: el `-1` es muy útil. Por ejemplo, para "aplanar" imágenes antes de pasarlas a un modelo clásico o a una capa densa:

In [19]:
# 60 imágenes de 28x28 píxeles (como el famoso dataset MNIST)
imagenes = np.zeros((60, 28, 28))
print("Antes:  ", imagenes.shape)

planas = imagenes.reshape(60, -1)     # cada imagen pasa a ser un vector de 784
print("Después:", planas.shape)

Antes:   (60, 28, 28)
Después: (60, 784)


In [20]:
# scikit-learn exige que X sea 2D: (n_muestras, n_features).
# Si tienes una sola feature como vector 1D, conviértela en columna:
x = np.array([1, 2, 3, 4])
print(x.shape)
print(x.reshape(-1, 1).shape)     # opción 1
print(x[:, np.newaxis].shape)     # opción 2 (equivalente)

(4,)
(4, 1)
(4, 1)


El dataset MNIST (Modified National Institute of Standards and Technology) es una base de datos pública y fundamental en el aprendizaje automático que contiene miles de imágenes de dígitos escritos a mano. Es considerado el equivalente al "Hola Mundo" en el campo de la visión por computadora.

Sus características principales son:

* **Contenido:** 70,000 imágenes de números del 0 al 9, originalmente recolectados de empleados de la Oficina del Censo de EE. UU. y estudiantes de secundaria.
* **Formato:** Cada imagen está en escala de grises, centrada, y tiene un tamaño muy pequeño y estandarizado de 28x28 píxeles.
* **Distribución:** El conjunto de datos está dividido por defecto en 60,000 imágenes para enseñar al modelo (conjunto de entrenamiento) y 10,000 imágenes para evaluar su precisión (conjunto de prueba).
* **Propósito:** Al ser un dataset limpio y fácil de procesar, se utiliza como un estándar de referencia rápido (benchmark) para que los desarrolladores e investigadores prueben algoritmos de clasificación de imágenes y arquitecturas de redes neuronales antes de escalar a problemas más pesados.

### 5.2 Transponer y aplanar

In [21]:
M = np.array([[1, 2, 3],
              [4, 5, 6]])
print("Original", M.shape, "\n", M)
print("Transpuesta", M.T.shape, "\n", M.T)
print("Aplanada:", M.ravel())

Original (2, 3) 
 [[1 2 3]
 [4 5 6]]
Transpuesta (3, 2) 
 [[1 4]
 [2 5]
 [3 6]]
Aplanada: [1 2 3 4 5 6]


El método `ravel()` de NumPy se utiliza para aplanar o "desenrollar" un arreglo multidimensional (como una matriz 2D o 3D) convirtiéndolo en un arreglo unidimensional (1D).

Sus características principales son:

* **Devuelve una vista (View):** Siempre que la estructura de memoria lo permita, `ravel()` no crea una copia nueva de los datos originales, sino que devuelve una "vista" plana del mismo arreglo. Esto lo hace muy eficiente en términos de memoria y velocidad de procesamiento.
* **Modifica el original:** Debido a que normalmente devuelve una vista, si cambias un valor en el arreglo resultante de `ravel()`, ese mismo valor cambiará en la matriz original.
* **Orden de lectura:** Por defecto, aplana los datos leyendo fila por fila (estilo C), pero puedes configurarlo para que lea columna por columna pasándole el argumento `order='F'` (estilo Fortran).

A menudo se compara con el método `flatten()`. Ambos hacen esencialmente lo mismo, pero `flatten()` **siempre** crea una copia completa de los datos en la memoria. Por lo tanto, `ravel()` es la opción preferida por rendimiento a menos que necesites aislar estrictamente los datos aplanados del arreglo original.

### 5.3 Unir arrays

In [22]:
a = np.array([[1, 2], [3, 4]])
b = np.array([[5, 6], [7, 8]])

print("Una debajo de otra (axis=0):\n", np.concatenate([a, b], axis=0))
print("Una al lado de otra (axis=1):\n", np.concatenate([a, b], axis=1))

# Atajos equivalentes
print(np.vstack([a, b]).shape, np.hstack([a, b]).shape)

Una debajo de otra (axis=0):
 [[1 2]
 [3 4]
 [5 6]
 [7 8]]
Una al lado de otra (axis=1):
 [[1 2 5 6]
 [3 4 7 8]]
(4, 2) (2, 4)


1. `np.concatenate` y `axis`
La función `np.concatenate` sirve para juntar o fusionar dos o más arreglos en uno solo. El parámetro `axis` (eje) es la clave aquí, ya que le indica a NumPy en qué dirección (o dimensión) debe realizar esa unión:

* **`axis=0` (Eje vertical o de filas):** Le indica a Python que una los arreglos de arriba hacia abajo. En el código, esto toma la matriz `b` y la coloca literalmente debajo de la matriz `a`. Por esto, el código imprime "Una debajo de otra". Si unes dos matrices de 2x2 de esta forma, terminas con una matriz más alta (4 filas y 2 columnas).


* **`axis=1` (Eje horizontal o de columnas):** Le indica a Python que una los arreglos de izquierda a derecha. Toma la matriz `b` y la pega al lado derecho de la matriz `a`. El código lo describe como "Una al lado de otra". El resultado es una matriz más ancha (2 filas y 4 columnas).



2. `vstack` y `hstack`
Como dice el comentario "# Atajos equivalentes" estas funciones son simplemente formas más rápidas y legibles de hacer exactamente lo mismo, sin tener que escribir ni recordar el número del `axis`:

* **`np.vstack([a, b])`:** Significa "Vertical Stack" (apilamiento vertical). Hace exactamente lo mismo que `np.concatenate([a, b], axis=0)`.


* **`np.hstack([a, b])`:** Significa "Horizontal Stack" (apilamiento horizontal). Hace exactamente lo mismo que `np.concatenate([a, b], axis=1)`.



Básicamente, `vstack` y `hstack` existen para que el código sea más fácil de leer y escribir sin preocuparte por los índices de los ejes.

In [23]:
# np.stack crea una NUEVA dimensión: útil para formar un "lote" (batch)
img1 = np.zeros((28, 28))
img2 = np.ones((28, 28))
lote = np.stack([img1, img2])
print(lote.shape)    # (2, 28, 28)

(2, 28, 28)


In [24]:
# Ejemplo ML: añadir una columna de unos a X (el "sesgo" o bias en regresión lineal)
X = np.array([[2.0, 3.0],
              [4.0, 5.0],
              [6.0, 7.0]])
X_con_bias = np.hstack([np.ones((X.shape[0], 1)), X])
print(X_con_bias)

[[1. 2. 3.]
 [1. 4. 5.]
 [1. 6. 7.]]


---
## 6. Operaciones vectorizadas

Las operaciones aritméticas se aplican elemento a elemento, sin bucles:

In [25]:
a = np.array([1, 2, 3, 4])
b = np.array([10, 20, 30, 40])

print("a + b =", a + b)
print("a * b =", a * b)
print("b / a =", b / a)
print("a ** 2 =", a ** 2)
print("a > 2 =", a > 2)

a + b = [11 22 33 44]
a * b = [ 10  40  90 160]
b / a = [10. 10. 10. 10.]
a ** 2 = [ 1  4  9 16]
a > 2 = [False False  True  True]


### Funciones universales (*ufuncs*)
NumPy incluye versiones vectorizadas de todas las funciones matemáticas:

In [26]:
x = np.array([0.0, 1.0, 2.0, 3.0])

print("exp:  ", np.exp(x))
print("log1p:", np.log1p(x))      # log(1 + x), evita log(0)
print("sqrt: ", np.sqrt(x))
print("sin:  ", np.sin(x))
print("abs:  ", np.abs(np.array([-3, 2, -1])))
print("clip: ", np.clip(np.array([-5, 0.5, 10]), 0, 1))  # recorta al rango [0, 1]

exp:   [ 1.          2.71828183  7.3890561  20.08553692]
log1p: [0.         0.69314718 1.09861229 1.38629436]
sqrt:  [0.         1.         1.41421356 1.73205081]
sin:   [0.         0.84147098 0.90929743 0.14112001]
abs:   [3 2 1]
clip:  [0.  0.5 1. ]


In [27]:
notas = np.array([[7.0, 8.5, 6.0],
                  [9.0, 9.5, 8.0],
                  [5.0, 6.0, 4.5],
                  [8.0, 7.0, 9.0]])

print("Media global:", notas.mean())

Media global: 7.291666666666667


Las funciones vectorizadas en NumPy aplican una operación matemática a cada elemento de un arreglo de forma individual y simultánea, sin necesidad de escribir ciclos `for`, esto es lo que hace cada función:

* **`np.exp(x)`:** Calcula la función exponencial (e^x) para cada elemento del arreglo. Por ejemplo, para el valor 1.0, devuelve la constante de Euler aproximada a 2.71828183.


* **`np.log1p(x)`:** Calcula el logaritmo natural de 1 más el valor del elemento (`log(1 + x)`). Como indica el comentario, esta función es útil porque evita el error matemático de intentar calcular el logaritmo de cero cuando el valor del elemento es 0.0.


* **`np.sqrt(x)`:** Calcula la raíz cuadrada (square root) de cada elemento. Por ejemplo, la raíz cuadrada de 2.0 resulta en 1.41421356.


* **`np.sin(x)`:** Calcula el seno trigonométrico de cada valor en el arreglo.


* **`np.abs()`:** Calcula el valor absoluto de los números, eliminando el signo negativo. En el ejemplo de la imagen, transforma el arreglo `[-3, 2, -1]` en `[3, 2, 1]`.


* **`np.clip(arreglo, minimo, maximo)`:** Recorta o limita los valores de un arreglo para que no se salgan de un rango específico. En la imagen, el rango establecido es `0, 1`. Por lo tanto, el valor -5 se ajusta a 0 (el mínimo), el 10 se ajusta a 1 (el máximo), y el 0.5 se mantiene igual porque ya está dentro del rango.


* **`notas.mean()`:** Calcula la media aritmética (el promedio) de los valores. En el código, se aplica a la matriz bidimensional `notas` para obtener la "Media global" (7.2916...) de todos los números contenidos en ella.

Para calcular el valor mínimo y máximo de un array en NumPy, puedes utilizar las funciones np.min() y np.max(), o bien los métodos propios del objeto array .min() y .max().
Aquí tienes una guía rápida de cómo usarlos:

1. En arrays de una dimensión (Vectores)

    Si tienes un array plano, estas funciones te devolverán un único valor.
    ```python
    import numpy as np

    # Crear un array de ejemplo
    arr = np.array([12, 5, 8, 22, 14, 3])

    # Opción A: Usando las funciones de NumPy
    minimo = np.min(arr)
    maximo = np.max(arr)

    # Opción B: Usando los métodos del array (más rápido de escribir)
    minimo_alt = arr.min()
    maximo_alt = arr.max()

    print(f"Mínimo: {minimo}")  # Resultado: 3
    print(f"Máximo: {maximo}")  # Resultado: 22
    ```

2. En arrays de dos dimensiones (Matrices)

    Cuando trabajas con matrices, puedes calcular el mínimo/máximo de tres formas distintas usando el argumento axis:
    - Todo el array: Si no especificas el eje, busca en toda la matriz.
    - Por columnas (axis=0): Compara los elementos verticalmente.
    - Por filas (axis=1): Compara los elementos horizontalmente.

    ```python
    import numpy as np

    # Creamos la matriz directamente asignando las filas
    fila_1 = [10, 20, 30]
    fila_2 = [5, 40, 15]
    matriz = np.array([fila_1, fila_2])

    # 1. Mínimo global de toda la matriz
    print(matriz.min())  
    # Resultado: 5

    # 2. Mínimo por columnas (axis=0) -> Compara verticalmente (10 vs 5, 20 vs 40, 30 vs 15)
    print(matriz.min(axis=0))  
    # Resultado: [5, 20, 15]

    # 3. Máximo por filas (axis=1) -> Busca el más alto de cada fila horizontalmente
    print(matriz.max(axis=1))  
    # Resultado: [30, 40]
    ```

3. Tip extra: ¿Quieres saber la posición (índice) en lugar del valor?

    Si lo que necesitas es saber en qué lugar del array se encuentra el número menor o mayor, utiliza np.argmin() y np.argmax():

    ```python
    arr = np.array([12, 5, 8, 22, 14, 3])

    indice_minimo = np.argmin(arr)
    indice_maximo = np.argmax(arr)

    print(f"El mínimo está en el índice: {indice_minimo}")  # Resultado: 5 (el valor 3)
    print(f"El máximo está en el índice: {indice_maximo}")  # Resultado: 3 (el valor 22)
    ```